# nmtc-mapper — NMTC Eligibility Checker
## Demo: Automated Eligibility Screening with Async Batch Geocoding

This notebook demonstrates how to use nmtc-mapper to:
- Check NMTC eligibility for individual addresses and census tracts
- Batch process thousands of addresses using async geocoding
- Analyze distress level distributions across a project portfolio
- Export results for IC memos and grant applications

Data source: CDFI Fund 2016-2020 ACS Low-Income Community Eligibility File
Geocoding: US Census Bureau Geocoding API (free, no API key required)


In [ ]:
import sys
sys.path.insert(0, '..')

from nmtcmapper import NMTCMapper
from nmtcmapper.data.loader import _build_sample_table
from nmtcmapper.eligibility.checker import enrich_dataframe
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

print("nmtc-mapper loaded successfully")


## 1. Initialize the Mapper

The mapper loads the CDFI Fund eligibility table on startup.
After the first run, it caches the file locally so subsequent
loads are instant.


In [ ]:
mapper = NMTCMapper()

print(f"Total census tracts loaded: {mapper.tract_count:,}")
print(f"NMTC eligible tracts:       {mapper.eligible_tract_count:,}")
print(f"Eligibility rate:           {mapper.eligible_tract_count/mapper.tract_count*100:.1f}%")


## 2. Check Individual Census Tracts

The fastest lookup — if you already know the 11-digit FIPS code,
no geocoding is needed.


In [ ]:
tracts_to_check = [
    ("17031840100", "Chicago South Side"),
    ("17031839100", "Chicago West Side"),
    ("17031010100", "Chicago North Shore"),
    ("26163518300", "Detroit"),
    ("36061015900", "NYC Bronx"),
    ("36047052200", "NYC Brooklyn Affluent"),
    ("13121010400", "Atlanta"),
    ("26001010100", "Rural Michigan"),
]

print(f"{'Tract ID':<15} {'Location':<25} {'Eligible':<10} {'Distress':<12} {'Poverty':<10} {'AMI Ratio'}")
print("-" * 85)

for tract_id, location in tracts_to_check:
    result = mapper.check_tract(tract_id)
    eligible = "YES" if result.nmtc_eligible else "NO"
    poverty = f"{result.poverty_rate*100:.1f}%" if result.poverty_rate else "N/A"
    ami = f"{result.ami_ratio*100:.1f}%" if result.ami_ratio else "N/A"
    print(f"{tract_id:<15} {location:<25} {eligible:<10} {result.distress_level:<12} {poverty:<10} {ami}")


## 3. Distress Level Analysis

Understanding distress levels is critical for NMTC allocation applications.
CDEs must commit at least 85% of resources to Severe Distress or
non-metro LIC tracts.


In [ ]:
table = _build_sample_table().reset_index()

print("Distress Level Distribution:")
print(table["distress_level"].value_counts().to_string())

print(f"\nBy distress level:")
for level in ["deep", "severe", "lic", "ineligible"]:
    subset = table[table["distress_level"] == level]
    if len(subset) > 0:
        avg_poverty = subset["poverty_rate"].mean()
        avg_ami = subset["ami_ratio"].mean()
        print(f"  {level:<12} {len(subset):>3} tracts | "
              f"avg poverty: {avg_poverty*100:.1f}% | "
              f"avg AMI: {avg_ami*100:.1f}%")


## 4. Batch Processing with Tract IDs

When you have a portfolio of projects with known census tract IDs,
enrich them all at once — no geocoding needed, instant results.


In [ ]:
portfolio = pd.DataFrame({
    "project_name": [
        "Southside Community Health Center",
        "North Shore Medical Office",
        "Detroit Advanced Manufacturing",
        "NYC Bronx Affordable Housing",
        "Chicago West Side Grocery",
        "Atlanta Community Center",
        "Brooklyn Luxury Condos",
        "Rural Michigan Food Bank",
    ],
    "project_type": [
        "Healthcare", "Healthcare", "Manufacturing",
        "Affordable Housing", "Food Access",
        "Community Facility", "Commercial RE", "Food Access",
    ],
    "amount": [
        3_500_000, 2_000_000, 9_500_000,
        6_000_000, 1_200_000, 2_800_000,
        8_000_000, 750_000,
    ],
    "tract_id": [
        "17031840100", "17031010100", "26163518300",
        "36061015900", "17031839100", "13121010400",
        "36047052200", "26001010100",
    ],
})

print(f"Portfolio: {len(portfolio)} projects, ${portfolio['amount'].sum()/1e6:.1f}MM total")
print()

enriched = mapper.enrich(portfolio, tract_col="tract_id")
print(enriched[["project_name", "nmtc_eligible", "distress_level",
                "poverty_rate", "ami_ratio"]].to_string(index=False))


## 5. Portfolio Eligibility Summary

In [ ]:
summary = mapper.eligible_count(enriched)

eligible_portfolio = enriched[enriched["nmtc_eligible"] == True]
print(f"Eligible project amount:  ${eligible_portfolio['amount'].sum()/1e6:.1f}MM")
print(f"Ineligible amount:        ${enriched[~enriched['nmtc_eligible']]['amount'].sum()/1e6:.1f}MM")
print(f"\nDistress level breakdown:")
print(enriched.groupby("distress_level")["amount"].agg(["count", "sum"]).to_string())


## 6. Async Batch Geocoding

For large address lists, nmtc-mapper uses async geocoding with:
- `asyncio` + `aiohttp` for concurrent requests
- Semaphore-based rate limiting (max 10 concurrent)
- Exponential backoff retry logic
- Progress bar via `tqdm`

**Note:** Real geocoding requires internet access and may take 1-2 seconds
per address. This demo shows the API — replace with real addresses to run.


In [ ]:
from nmtcmapper.geocoder.census import (
    geocode_address, _parse_street, _parse_city, _parse_state, _parse_zip
)

# Show address parsing
sample_address = "1234 S Michigan Ave, Chicago, IL 60605"
print(f"Full address:  {sample_address}")
print(f"Street:        {_parse_street(sample_address)}")
print(f"City:          {_parse_city(sample_address)}")
print(f"State:         {_parse_state(sample_address)}")
print(f"ZIP:           {_parse_zip(sample_address)}")
print()
print("To geocode a real address:")
print("  tract_id = geocode_address('1234 S Michigan Ave, Chicago, IL 60605')")
print()
print("To batch geocode a DataFrame:")
print("  df = mapper.enrich(df, address_col='address')")
print("  # Uses async processing: 10 concurrent requests, retry on failure")


## 7. Eligibility by Project Type

In [ ]:
type_summary = enriched.groupby("project_type").agg(
    count=("amount", "count"),
    total_amount=("amount", "sum"),
    nmtc_eligible=("nmtc_eligible", "sum"),
).reset_index()

type_summary["pct_eligible"] = (
    type_summary["nmtc_eligible"] / type_summary["count"] * 100
).round(1)

print("Portfolio by Project Type:")
print(type_summary.to_string(index=False))


## 8. Export Results

In [ ]:
import tempfile, os

with tempfile.TemporaryDirectory() as tmpdir:
    csv_path = os.path.join(tmpdir, "nmtc_eligibility_results.csv")
    enriched.to_csv(csv_path, index=False)
    print(f"Exported {len(enriched)} rows to CSV")

    reloaded = pd.read_csv(csv_path)
    print(f"Columns exported: {list(reloaded.columns)}")
    print(f"\nEligible projects in export:")
    eligible = reloaded[reloaded["nmtc_eligible"] == True]
    print(eligible[["project_name", "distress_level", "poverty_rate"]].to_string(index=False))


## Summary

This notebook demonstrated the full nmtc-mapper workflow:

1. **Single tract lookup** — instant eligibility check by FIPS code
2. **Distress level analysis** — deep, severe, LIC, ineligible classification
3. **Batch enrichment** — enrich 10,000 rows in seconds using tract IDs
4. **Portfolio analysis** — eligibility and amount breakdowns by type
5. **Async geocoding** — convert addresses to tract IDs at scale
6. **Export** — CSV output for IC memos and grant applications

**Key advantage:** What previously required manual lookups in the CDFI Fund
CIMS tool one address at a time can now be done programmatically across
an entire portfolio in seconds.

**GitHub:** https://github.com/Jaypatel1511/nmtc-mapper
**Docs:** https://jaypatel1511.github.io/nmtc-mapper
**PyPI:** https://pypi.org/project/nmtc-mapper
